# 02 - Feature engineering
Two grains, built in `src/features.py`: **application** (7,500 rows, for the default model) and **customer** (5,000 rows). Transactions are aggregated to one row per customer *before* the join - the PRD's duplication warning.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, json
sns.set_theme(style='whitegrid')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
from src import features as fe
t = fe.load_tables(); a = fe.application_features(t); c = fe.customer_features(t)
print('application grain', a.shape, '| customer grain', c.shape)
print('rows in application table', len(a), '== loan_applications rows', len(t['loan_applications']))
assert len(a) == len(t['loan_applications']) and a.application_id.is_unique   # no join fan-out
tx = fe.transaction_features(t['transactions']); print('transaction aggregate: one row per customer ->', tx.customer_id.is_unique, len(tx))
tx.describe().T.round(1)

application grain (7500, 64) | customer grain (5000, 49)
rows in application table 7500 == loan_applications rows 7500
transaction aggregate: one row per customer -> True 5000


,count,mean,std,min,25%,50%,75%,max
txn_count,5000.0,8.0,2.6,4.0,6.0,8.0,10.0,12.0
total_inflow,5000.0,59233.9,69523.5,0.0,11513.9,38584.5,82953.0,695246.8
total_outflow,5000.0,53313.6,46359.8,0.0,21976.1,39555.7,69634.9,478836.7
emi_outflow,5000.0,8365.7,13125.5,0.0,0.0,3254.4,11933.6,147367.8
avg_balance,5000.0,80419.7,62152.1,6595.2,38847.0,63889.3,100614.3,629632.6
min_balance,5000.0,25247.8,30083.3,100.0,7605.1,15934.2,31054.8,391918.6
net_cash_flow,5000.0,5920.3,64520.8,-355253.7,-24048.9,925.5,30295.9,486982.2
inflow_outflow_ratio,4998.0,1.5,3.2,0.0,0.4,1.0,1.9,168.6
emi_share_of_outflow,4998.0,0.2,0.2,0.0,0.0,0.1,0.3,1.0


## Transactions are sparse
Only 4-12 transactions per customer over six months (mean 8). The aggregated cash-flow features are therefore *thin* - one salary credit can change `inflow_outflow_ratio` dramatically, which is why numeric inputs are winsorised.

In [2]:
print(t['transactions'].groupby('customer_id').size().describe().round(1).to_dict())
print(tx[['inflow_outflow_ratio', 'emi_share_of_outflow']].quantile([0, .5, .99, 1]).round(2))

{'count': 5000.0, 'mean': 8.0, 'std': 2.6, 'min': 4.0, '25%': 6.0, '50%': 8.0, '75%': 10.0, 'max': 12.0}
      inflow_outflow_ratio  emi_share_of_outflow
0.00                  0.00                  0.00
0.50                  1.03                  0.10
0.99                  9.62                  0.72
1.00                168.64                  1.00


## Leakage control
Excluded: `approval_status`, `approved_amount`, `decision_date`, `disbursed_flag` (known only after the decision), contact details and identifiers (PII), `gender`/`marital_status` (not in the PRD feature list; fair-lending review is out of scope), city/state (dashboard only).

In [3]:
num, cat = fe.NUM_APP, fe.CATEGORICAL_APP
banned = ['approval_status', 'approved_amount', 'decision_date', 'disbursed_flag', 'email', 'phone', 'first_name', 'last_name', 'gender', 'marital_status', 'loan_default']
X, params = fe.design_matrix(a[a.loan_default.notna()], num, cat)
print(X.shape[1], 'model inputs;', 'forbidden columns present:', [c_ for c_ in X.columns if any(c_.startswith(b) for b in banned)])
assert not [c_ for c_ in X.columns if any(c_.startswith(b) for b in banned)]

55 model inputs; forbidden columns present: []


## The caveat that cannot be engineered away
The bureau snapshot is dated 2024-06-30, but applications span Jul 2022 - Jun 2024, and the transaction window is Jan-Jun 2024. For older loans these features were measured *after* repayment behaviour may already have changed them. We test whether this inflates results (notebook 03: AUC by application year, with and without bureau features) but cannot rule it out.

In [4]:
l = t['loan_applications']; print('applications before 2024-01-01 (transaction window starts):', int((l.application_date < '2024-01-01').sum()), 'of', len(l))
early = a.merge(t['customers'][['customer_id', 'customer_since']], on='customer_id')
print('applications dated BEFORE customer_since:', int((pd.to_datetime(early.customer_since_x if 'customer_since_x' in early else early.customer_since) > pd.to_datetime(early.application_date)).sum()))

applications before 2024-01-01 (transaction window starts): 5681 of 7500
applications dated BEFORE customer_since: 935


In [5]:
# winsorisation effect on the two heavy-tailed ratios
raw_q = a[['loan_to_income', 'inflow_outflow_ratio']].quantile([.5, .99, 1]).round(1)
Xc, _ = fe.design_matrix(a, num, cat); print(raw_q.T, '\nafter winsorising max:', Xc[['loan_to_income', 'inflow_outflow_ratio']].max().round(1).to_dict())

                      0.50  0.99   1.00
loan_to_income         2.8  49.8  100.0
inflow_outflow_ratio   1.0   9.9  168.6 
after winsorising max: {'loan_to_income': 49.8, 'inflow_outflow_ratio': 9.9}
